In [9]:
# --------------------------------------------
# Oracle Fusion Invoice Extract + OpenAI Q&A
# --------------------------------------------

import pandas as pd
import requests
import getpass
from openai import OpenAI

# Step 1. Secure runtime inputs (no secrets in code)
fusion_url = input("Enter your Oracle Fusion base URL (e.g. https://abcd.fa.us6.oraclecloud.com): ").strip()
username = input("Enter your Oracle Fusion username: ").strip()
password = getpass.getpass("Enter your Oracle Fusion password: ").strip()
openai_api_key = getpass.getpass("Enter your OpenAI API key: ").strip()

# Initialize OpenAI client
client = OpenAI(api_key=openai_api_key)

# Step 2. Oracle REST endpoint for Receivables Invoices
endpoint = f"{fusion_url}/fscmRestApi/resources/11.13.18.05/receivablesInvoices"

# Optional parameters
params = {"limit": 100}  # fetch first 100; pagination can be added later

# Step 3. Call Oracle REST API
print("\nConnecting to Oracle Fusion ...")
response = requests.get(endpoint, auth=(username, password), params=params)

if response.status_code != 200:
    print("❌ Connection failed. Please check credentials, endpoint, or network.")
    print("Status Code:", response.status_code)
    print("Response:", response.text)
else:
    print("✅ Connected successfully!")

    data = response.json().get('items', [])
    if not data:
        print("No data returned from API.")
    else:
        # Step 4. Convert JSON → pandas DataFrame
        df = pd.json_normalize(data)

        # Select & rename useful columns (adjust to your payload fields)
        columns_map = {
            'InvoiceNumber': 'Invoice Number',
            'InvoiceDate': 'Invoice Date',
            'CurrencyCode': 'Currency',
            'CustomerTrxTypeName': 'Customer Name',
            'DueDate': 'Due Date',
            'PaymentTerms': 'Payment Term',
            'BusinessUnit': 'Business Unit',
            'Status': 'Status'
        }

        available_cols = [col for col in columns_map.keys() if col in df.columns]
        display_df = df[available_cols].copy()
        display_df.rename(columns=columns_map, inplace=True)

        print("\n📊 Sample Invoices:")
        display(display_df.head())

        # Step 5. Define a Natural Language Q&A helper
        def ask_invoice_question(question: str):
            """
            Send a question about the current invoice dataset to OpenAI for analysis.
            The model will only use data visible in display_df.
            """
            sample = display_df.head(50).to_string(index=False)
            prompt = f"""
            You are an Oracle Receivables analyst.
            Here is a sample of invoice data extracted from Oracle Fusion Cloud:

            {sample}

            Using only the data above, answer the following question:
            {question}
            """

            completion = client.chat.completions.create(
                model="gpt-4-turbo",
                messages=[
                    {"role": "system", "content": "You analyze Oracle Receivables invoices."},
                    {"role": "user", "content": prompt}
                ]
            )
            return completion.choices[0].message.content

        print("\n✅ Ready for Q&A! Try:")
        print("ask_invoice_question('Which invoices are overdue?')")


Enter your Oracle Fusion base URL (e.g. https://abcd.fa.us6.oraclecloud.com):  https://dabpqy.ds-fa.oraclepdemos.com
Enter your Oracle Fusion username:  MTOBWY
Enter your Oracle Fusion password:  ········
Enter your OpenAI API key:  ········



Connecting to Oracle Fusion ...
✅ Connected successfully!

📊 Sample Invoices:


,Due Date,Payment Term,Business Unit
0,2012-03-26,Net 15,UK Business Unit
1,2012-08-03,Net 30,UK Business Unit
2,2012-07-28,Net 30,UK Business Unit
3,2012-07-29,Net 30,UK Business Unit
4,2012-05-06,Net 30,UK Business Unit



✅ Ready for Q&A! Try:
ask_invoice_question('Which invoices are overdue?')


In [10]:
question = input("Ask a question about the invoices: ")
answer = ask_invoice_question(question)
print("\n💡 Answer:\n", answer)


Ask a question about the invoices:  give me 1 invoice from US1 Business Unit



💡 Answer:
 Based on the data provided, all the invoices are from the "UK Business Unit." There are no invoices listed for the "US1 Business Unit." Therefore, I cannot provide an invoice from the US1 Business Unit using only this data.
